In [1]:
# --- 0.1 Import library ---
import os, io, re, json, time, shutil, hashlib, datetime as dt
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import numpy as np
import pandas as pd
import requests

# --- 0.2 Identitas & konfigurasi (email kampus dipakai di User-Agent; bukan API key) ---
NAMA, NIM = "I Gusti Ngurah Dwi Andika Putra", "2501010016"
EMAIL = "igustingurahdwiandikaputra@instiki.ac.id"

TIMEOUT = 20   # detik; wajib di setiap request
DELAY = 2      # jeda antar-request (detik)
HEADERS = {"User-Agent": f"INSTIKI-UTS-DAM/1.0 (mahasiswa; {EMAIL})"}

# --- 0.3 Struktur folder ---
for folder in ["data/raw", "data/processed", "metadata", "evidence", "docs"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# --- 0.4 Berkas & URL sumber ---
FILE_RAW = "data/raw/dataset_wisata_cuaca_bali_2026.xlsx"
URL_DTW_1 = "https://data.go.id/dataset/dataset/sebaran-daya-tarik-wisata-di-bali"
URL_DTW_2 = "https://data.go.id/dataset/dataset/data-daya-tarik-wisata-alam-di-provinsi-bali"
URL_OPENMETEO_DOC = "https://open-meteo.com/en/docs/historical-weather-api"
URL_OPENMETEO_TERMS = "https://open-meteo.com/en/terms"
API_URL = "https://archive-api.open-meteo.com/v1/archive"
PAGE_URL = "https://en.wikipedia.org/wiki/List_of_districts_in_Bali"
print("Setup selesai.", HEADERS["User-Agent"])

Setup selesai. INSTIKI-UTS-DAM/1.0 (mahasiswa; igustingurahdwiandikaputra@instiki.ac.id)


In [2]:
# --- 0.5 Fungsi bantu: MD5, request aman (timeout, UA, jeda, retry), provenance, cek robots ---
def md5_file(path):
    """Hash MD5 sebuah berkas."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_get(url, params=None, max_retry=3):
    """GET dengan timeout, User-Agent, jeda, penanganan status code, dan retry (exponential backoff)."""
    for attempt in range(1, max_retry + 1):
        try:
            time.sleep(DELAY)                                              # jeda antar-request
            r = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
            print(f"[{attempt}] HTTP {r.status_code} <- {r.url}")
            if r.status_code == 200:
                return r
            if r.status_code in (429, 500, 502, 503, 504):                 # error sementara -> coba lagi
                time.sleep(2 ** attempt)
                continue
            print("    Respons:", r.text[:200])                            # 4xx lain: tampilkan alasan, hentikan
            r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[{attempt}] Gagal koneksi: {type(e).__name__}")
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Gagal mengambil {url} setelah {max_retry} percobaan")

PROV_PATH = "metadata/provenance.csv"
provenance = {}   # kunci: path berkas raw
def log_provenance(sumber, cara, url, path_raw, lisensi, catatan=""):
    """Catat provenance (URL, tanggal akses, lisensi, MD5) dan simpan ke CSV."""
    provenance[path_raw] = {
        "sumber": sumber, "cara_akuisisi": cara, "url": url,
        "tanggal_akses": dt.date.today().isoformat(), "lisensi": lisensi,
        "berkas_raw": path_raw, "ukuran_byte": os.path.getsize(path_raw),
        "md5": md5_file(path_raw), "catatan": catatan,
    }
    pd.DataFrame(provenance.values()).to_csv(PROV_PATH, index=False)

def cek_robots(url):
    """Unduh robots.txt (timeout+UA), simpan sebagai bukti, dan uji can_fetch untuk User-Agent kita."""
    p = urlparse(url)
    robots_url = f"{p.scheme}://{p.netloc}/robots.txt"
    try:
        time.sleep(DELAY)
        r = requests.get(robots_url, headers=HEADERS, timeout=TIMEOUT)
        Path(f"evidence/robots_{p.netloc.replace('.', '_')}.txt").write_text(r.text, encoding="utf-8")
        rp = RobotFileParser()
        # 404 = tidak ada robots.txt (tidak ada larangan teknis); 200 = aturan dibaca; lainnya = tidak pasti
        rp.parse(r.text.splitlines() if r.status_code == 200 else [])
        boleh = rp.can_fetch(HEADERS["User-Agent"], url) if r.status_code in (200, 404) else None
        return {"robots_url": robots_url, "status": r.status_code, "boleh": boleh,
                "crawl_delay": rp.crawl_delay(HEADERS["User-Agent"]),
                "dicek": dt.datetime.now().isoformat(timespec="seconds")}
    except requests.RequestException as e:
        return {"robots_url": robots_url, "status": None, "boleh": None, "crawl_delay": None,
                "dicek": f"GAGAL ({type(e).__name__}); ulangi saat online"}

In [3]:
# --- 0.6 Siapkan berkas dataset di data/raw (Colab: unggah manual jika belum ada) ---
NAMA_BERKAS = os.path.basename(FILE_RAW)
if not os.path.exists(FILE_RAW):
    if os.path.exists(NAMA_BERKAS):                      # sudah ada di folder kerja -> pindahkan
        shutil.move(NAMA_BERKAS, FILE_RAW)
    else:
        try:
            from google.colab import files               # khusus Google Colab
            print(f"Unggah berkas {NAMA_BERKAS} ...")
            for fn in files.upload().keys():
                shutil.move(fn, "data/raw/" + fn)
        except ImportError:
            pass
assert os.path.exists(FILE_RAW), f"Letakkan {NAMA_BERKAS} di folder data/raw"
print("Berkas siap:", FILE_RAW, f"({os.path.getsize(FILE_RAW)} byte)")

Unggah berkas dataset_wisata_cuaca_bali_2026.xlsx ...


Saving dataset_wisata_cuaca_bali_2026.xlsx to dataset_wisata_cuaca_bali_2026.xlsx
Berkas siap: data/raw/dataset_wisata_cuaca_bali_2026.xlsx (134776 byte)


In [4]:
# --- 1.1 Lihat isi workbook: nama sheet dan ukurannya ---
xls = pd.ExcelFile(FILE_RAW)
for s in xls.sheet_names:
    d = pd.read_excel(FILE_RAW, sheet_name=s)
    print(f"{s:24s} baris={d.shape[0]:5d} kolom={d.shape[1]}")

Wisata_108               baris=  108 kolom=9
Cuaca_Harian_Template    baris= 2880 kolom=12
Master_Gabungan          baris=  108 kolom=11
Sumber_dan_Lisensi       baris=    4 kolom=6
Proses_Akuisisi          baris=    5 kolom=4
URL_API_Cuaca            baris=    9 kolom=4


In [5]:
# --- 1.2 Baca sheet utama dengan parameter yang tepat ---
# sheet_name -> sheet daftar objek wisata; dtype=str untuk id/nama (identifier, bukan angka);
# na_values -> penanda sel kosong; koordinat dibaca sebagai float
df = pd.read_excel(
    FILE_RAW,
    sheet_name="Wisata_108",
    dtype={"id_wisata": str, "nama_objek_wisata": str, "kabupaten_kota": str, "jenis_wisata": str},
    na_values=["", "-", "NA", "N/A"],
)
df.columns = df.columns.str.strip()
for c in ["nama_objek_wisata", "kabupaten_kota", "jenis_wisata"]:
    df[c] = df[c].str.strip()                            # rapikan spasi (di salinan, bukan di berkas raw)
print(df.shape)
df.info()
df.head()

(108, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_wisata               108 non-null    object 
 1   nama_objek_wisata       108 non-null    object 
 2   kabupaten_kota          108 non-null    object 
 3   jenis_wisata            108 non-null    object 
 4   latitude_kabupaten      108 non-null    float64
 5   longitude_kabupaten     108 non-null    float64
 6   tahun_referensi_daftar  108 non-null    object 
 7   status_verifikasi       108 non-null    object 
 8   sumber_data             108 non-null    object 
dtypes: float64(2), object(7)
memory usage: 7.7+ KB


,id_wisata,nama_objek_wisata,kabupaten_kota,jenis_wisata,latitude_kabupaten,longitude_kabupaten,tahun_referensi_daftar,status_verifikasi,sumber_data
0,BALI001,Pantai Kuta,Badung,Alam,-8.65,115.18,2019/2021 dan kompilasi nama destinasi,Perlu verifikasi ke dinas/portal resmi untuk s...,Pemprov Bali / Data.go.id sebagai referensi aw...
1,BALI002,Pantai Legian,Badung,Alam,-8.65,115.18,2019/2021 dan kompilasi nama destinasi,Perlu verifikasi ke dinas/portal resmi untuk s...,Pemprov Bali / Data.go.id sebagai referensi aw...
2,BALI003,Pantai Seminyak,Badung,Alam,-8.65,115.18,2019/2021 dan kompilasi nama destinasi,Perlu verifikasi ke dinas/portal resmi untuk s...,Pemprov Bali / Data.go.id sebagai referensi aw...
3,BALI004,Pantai Petitenget,Badung,Alam,-8.65,115.18,2019/2021 dan kompilasi nama destinasi,Perlu verifikasi ke dinas/portal resmi untuk s...,Pemprov Bali / Data.go.id sebagai referensi aw...
4,BALI005,Pantai Berawa,Badung,Alam,-8.65,115.18,2019/2021 dan kompilasi nama destinasi,Perlu verifikasi ke dinas/portal resmi untuk s...,Pemprov Bali / Data.go.id sebagai referensi aw...


In [6]:
# --- 1.3 Identifikasi dataset ---
ukuran_byte = os.path.getsize(FILE_RAW)
identitas = pd.Series({
    "Jenis data": "Terstruktur (tabel baris-kolom pada Excel)",
    "Jumlah baris (objek wisata)": df.shape[0],
    "Jumlah kolom": df.shape[1],
    "Satuan": "Koordinat: derajat desimal (lintang/bujur); kolom lain bersifat nominal (teks/kategori)",
    "Cakupan wilayah": f"{df['kabupaten_kota'].nunique()} kabupaten/kota di Provinsi Bali",
    "Cakupan waktu": "Daftar rujukan tahun 2019/2021 (kompilasi); tidak memuat data deret waktu",
    "Ukuran berkas": f"{ukuran_byte} byte ({ukuran_byte/1024:.2f} KB)",
    "Nilai kosong": int(df.isna().sum().sum()),
    "ID unik": bool(df['id_wisata'].is_unique),
})
display(identitas.to_frame("Nilai"))
print("\nTipe data per kolom:")
display(df.dtypes.astype(str).to_frame("tipe"))
print("Objek per kabupaten/kota:")
display(df["kabupaten_kota"].value_counts().to_frame("jumlah"))

,Nilai
Jenis data,Terstruktur (tabel baris-kolom pada Excel)
Jumlah baris (objek wisata),108
Jumlah kolom,9
Satuan,Koordinat: derajat desimal (lintang/bujur); ko...
Cakupan wilayah,9 kabupaten/kota di Provinsi Bali
Cakupan waktu,Daftar rujukan tahun 2019/2021 (kompilasi); ti...
Ukuran berkas,134776 byte (131.62 KB)
Nilai kosong,0
ID unik,True



Tipe data per kolom:


,tipe
id_wisata,object
nama_objek_wisata,object
kabupaten_kota,object
jenis_wisata,object
latitude_kabupaten,float64
longitude_kabupaten,float64
tahun_referensi_daftar,object
status_verifikasi,object
sumber_data,object


Objek per kabupaten/kota:


,jumlah
kabupaten_kota,
Badung,12
Bangli,12
Buleleng,12
Denpasar,12
Gianyar,12
Jembrana,12
Karangasem,12
Klungkung,12
Tabanan,12


In [7]:
# --- 1.4 Estimasi kebutuhan storage 1 tahun: ukuran x 12 bulan + cadangan 30% ---
ukuran_mb = ukuran_byte / (1024 * 1024)
setahun_mb = ukuran_mb * 12
total_mb = setahun_mb * 1.30
print(f"Ukuran berkas          : {ukuran_mb:.4f} MB")
print(f"x 12 bulan             : {setahun_mb:.4f} MB")
print(f"+ cadangan 30%         : {total_mb:.4f} MB (= {total_mb*1024:.1f} KB)")

Ukuran berkas          : 0.1285 MB
x 12 bulan             : 1.5424 MB
+ cadangan 30%         : 2.0051 MB (= 2053.2 KB)


In [8]:
# --- 1.5 Catat provenance berkas dataset ---
# Penting: berkas ini adalah kompilasi/kurasi yang merujuk daftar DTW Pemprov Bali di data.go.id
# (lihat kolom sumber_data & status_verifikasi), bukan salinan asli portal. Hal ini dicatat apa adanya.
catatan_sumber = pd.read_excel(FILE_RAW, sheet_name="Wisata_108")["sumber_data"].iloc[0]
log_provenance(
    sumber="Daftar objek wisata Bali (berkas XLSX, sheet Wisata_108)",
    cara="Berkas (diunduh/disusun manual)",
    url=f"{URL_DTW_1} ; {URL_DTW_2}",
    path_raw=FILE_RAW,
    lisensi="Tidak dicantumkan pada halaman dataset data.go.id (diperiksa pada tanggal akses)",
    catatan=f"Kompilasi merujuk portal. Keterangan di berkas: '{catatan_sumber}'. Status: perlu verifikasi ke dinas/portal resmi.",
)
display(pd.DataFrame(provenance.values()).T)

,0
sumber,"Daftar objek wisata Bali (berkas XLSX, sheet W..."
cara_akuisisi,Berkas (diunduh/disusun manual)
url,https://data.go.id/dataset/dataset/sebaran-day...
tanggal_akses,2026-11-18
lisensi,Tidak dicantumkan pada halaman dataset data.go...
berkas_raw,data/raw/dataset_wisata_cuaca_bali_2026.xlsx
ukuran_byte,134776
md5,47152b5e47620f3a4628bdacdaf2def4
catatan,Kompilasi merujuk portal. Keterangan di berkas...


In [9]:
# --- 2.1 Rekap lisensi & ketentuan tiap sumber ---
SUMBER = [
    {"jenis": "berkas", "nama": "Daftar DTW Bali (data.go.id, Pemprov Bali)",
     "url": f"{URL_DTW_1} ; {URL_DTW_2}",
     "lisensi": "Tidak dicantumkan pada halaman dataset (dicek manual pada tanggal akses)",
     "ketentuan": "Portal data terbuka pemerintah; tanpa lisensi eksplisit dipakai secara konservatif: tujuan akademik non-komersial, mencantumkan sumber, tidak mempublikasikan ulang sebagai data baru."},
    {"jenis": "api", "nama": "Open-Meteo Historical Weather API",
     "url": API_URL,
     "lisensi": "CC BY 4.0 (atribusi wajib: 'Weather data by Open-Meteo.com')",
     "ketentuan": "API gratis non-komersial; kurang dari 10.000 panggilan/hari, 5.000/jam, 600/menit (halaman terms Open-Meteo)."},
    {"jenis": "scraping", "nama": "Wikipedia: List of districts in Bali",
     "url": PAGE_URL,
     "lisensi": "CC BY-SA 4.0 (atribusi + share-alike)",
     "ketentuan": "Halaman statis publik; robots.txt diperiksa; User-Agent berisi kontak; jeda antar-request."},
]
display(pd.DataFrame(SUMBER)[["jenis", "nama", "lisensi"]])

,jenis,nama,lisensi
0,berkas,"Daftar DTW Bali (data.go.id, Pemprov Bali)",Tidak dicantumkan pada halaman dataset (dicek ...
1,api,Open-Meteo Historical Weather API,CC BY 4.0 (atribusi wajib: 'Weather data by Op...
2,scraping,Wikipedia: List of districts in Bali,CC BY-SA 4.0 (atribusi + share-alike)


In [10]:
# --- 2.2 Bukti lisensi: simpan salinan halaman portal ke evidence/ (jika portal menolak akses otomatis, pakai tangkapan layar manual) ---
bukti_lisensi = []
halaman_bukti = {"datagoid_sebaran_dtw": URL_DTW_1, "datagoid_dtw_alam": URL_DTW_2,
                 "openmeteo_terms": URL_OPENMETEO_TERMS, "openmeteo_docs": URL_OPENMETEO_DOC}
for nama, url in halaman_bukti.items():
    try:
        r = safe_get(url, max_retry=2)
        Path(f"evidence/lisensi_{nama}.html").write_text(r.text, encoding="utf-8")
        bukti_lisensi.append(f"{nama}: tersimpan di evidence/lisensi_{nama}.html")
    except Exception as e:
        bukti_lisensi.append(f"{nama}: akses otomatis gagal ({type(e).__name__}); gunakan tangkapan layar manual di evidence/")
print("\n".join(bukti_lisensi))

[1] HTTP 404 <- https://data.go.id/dataset/dataset/sebaran-daya-tarik-wisata-di-bali
    Respons: <!DOCTYPE html><html id="__next_error__"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="script" fetchPriority="low" href="/_n
[1] HTTP 404 <- https://data.go.id/dataset/dataset/data-daya-tarik-wisata-alam-di-provinsi-bali
    Respons: <!DOCTYPE html><html id="__next_error__"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="script" fetchPriority="low" href="/_n
[1] HTTP 200 <- https://open-meteo.com/en/terms
[1] HTTP 200 <- https://open-meteo.com/en/docs/historical-weather-api
datagoid_sebaran_dtw: akses otomatis gagal (HTTPError); gunakan tangkapan layar manual di evidence/
datagoid_dtw_alam: akses otomatis gagal (HTTPError); gunakan tangkapan layar manual di evidence/
openmeteo_terms: tersimpan di evidence/lisensi_openmeteo_terms.html
openmete

In [11]:
# --- 2.3 Identifikasi PII / quasi-identifier ---
KATA_PII = ["nama_pribadi", "email", "telepon", "phone", "hp", "nik", "ktp", "alamat", "address", "rekening", "foto"]
KATA_QI = ["lahir", "birth", "usia", "umur", "kode_pos", "zip", "jenis_kelamin", "gender", "pekerjaan"]
POLA = {"email": r"^[^@\s]+@[^@\s]+\.[a-z]{2,}$", "telepon": r"^(\+?\d{1,3}|0)8?\d{8,12}$", "nik": r"^\d{16}$"}

def deteksi_pii(data):
    """Usulan klasifikasi per kolom: dari nama kolom dan pola isi (email/telepon/NIK)."""
    hasil = []
    for kol in data.columns:
        nilai = data[kol].dropna().astype(str).head(50)
        pola = [n for n, p in POLA.items() if len(nilai) and nilai.str.match(p, case=False).mean() > 0.8]
        low = kol.lower()
        if pola or any(k in low for k in KATA_PII):
            kategori = "PII langsung"
        elif any(k in low for k in KATA_QI):
            kategori = "Quasi-identifier"
        else:
            kategori = "Non-PII"
        hasil.append({"kolom": kol, "kategori": kategori, "pola_isi": ", ".join(pola) or "-"})
    return pd.DataFrame(hasil)

klasifikasi = deteksi_pii(df)
# Alasan per kolom (diisi manual setelah memeriksa isi kolom)
ALASAN = {
    "id_wisata": "Kode urut internal objek wisata, bukan identitas orang",
    "nama_objek_wisata": "Nama tempat/objek wisata (pura, pantai, museum, desa wisata), bukan nama individu",
    "kabupaten_kota": "Wilayah administratif tingkat kabupaten/kota; terlalu umum untuk mengidentifikasi individu",
    "jenis_wisata": "Kategori objek wisata",
    "latitude_kabupaten": "Koordinat representatif kabupaten/kota (bukan lokasi rumah/individu)",
    "longitude_kabupaten": "Koordinat representatif kabupaten/kota (bukan lokasi rumah/individu)",
    "tahun_referensi_daftar": "Keterangan tahun rujukan dataset",
    "status_verifikasi": "Catatan status verifikasi dataset",
    "sumber_data": "Keterangan sumber dataset",
}
klasifikasi["alasan"] = klasifikasi["kolom"].map(ALASAN)
display(klasifikasi)

pii_kolom = klasifikasi.loc[klasifikasi["kategori"] == "PII langsung", "kolom"].tolist()
qi_kolom = klasifikasi.loc[klasifikasi["kategori"] == "Quasi-identifier", "kolom"].tolist()
print("PII langsung    :", pii_kolom or "tidak ada")
print("Quasi-identifier:", qi_kolom or "tidak ada")

,kolom,kategori,pola_isi,alasan
0,id_wisata,Non-PII,-,"Kode urut internal objek wisata, bukan identit..."
1,nama_objek_wisata,Non-PII,-,"Nama tempat/objek wisata (pura, pantai, museum..."
2,kabupaten_kota,Non-PII,-,Wilayah administratif tingkat kabupaten/kota; ...
3,jenis_wisata,Non-PII,-,Kategori objek wisata
4,latitude_kabupaten,Non-PII,-,Koordinat representatif kabupaten/kota (bukan ...
5,longitude_kabupaten,Non-PII,-,Koordinat representatif kabupaten/kota (bukan ...
6,tahun_referensi_daftar,Non-PII,-,Keterangan tahun rujukan dataset
7,status_verifikasi,Non-PII,-,Catatan status verifikasi dataset
8,sumber_data,Non-PII,-,Keterangan sumber dataset


PII langsung    : tidak ada
Quasi-identifier: tidak ada


In [12]:
# --- 2.4 Penanganan PII (kosong bila tidak ada PII langsung) ---
SALT = "salt-uts-dam"   # hanya dipakai jika perlu pseudonimisasi
def pseudonim(v, panjang=12):
    return hashlib.sha256((SALT + str(v)).encode()).hexdigest()[:panjang]

TINDAKAN = {}   # contoh bila ada PII: {"kolom": "hapus"} atau {"kolom": "pseudonim"}
aman = df.copy()
for kol, aksi in TINDAKAN.items():
    aman = aman.drop(columns=kol) if aksi == "hapus" else aman.assign(**{kol: aman[kol].map(pseudonim)})
sisa_pii = deteksi_pii(aman).query("kategori == 'PII langsung'")
print("Sisa PII setelah penanganan:", sisa_pii["kolom"].tolist() or "tidak ada")

# --- 2.5 Minimalisasi data: hanya kolom yang dibutuhkan untuk tujuan analisis ---
KOLOM_DIPAKAI = ["id_wisata", "nama_objek_wisata", "kabupaten_kota", "jenis_wisata", "latitude_kabupaten", "longitude_kabupaten"]
tidak_dipakai = [k for k in aman.columns if k not in KOLOM_DIPAKAI]
print("Dipakai :", KOLOM_DIPAKAI)
print("Dibuang :", tidak_dipakai, "(isinya konstan; dicatat di metadata/provenance)")

Sisa PII setelah penanganan: tidak ada
Dipakai : ['id_wisata', 'nama_objek_wisata', 'kabupaten_kota', 'jenis_wisata', 'latitude_kabupaten', 'longitude_kabupaten']
Dibuang : ['tahun_referensi_daftar', 'status_verifikasi', 'sumber_data'] (isinya konstan; dicatat di metadata/provenance)


In [13]:
# --- 2.6 Pemeriksaan robots.txt untuk sumber scraping (bukti disimpan di evidence/) ---
robots_wiki = cek_robots(PAGE_URL)
print(robots_wiki)
json.dump(robots_wiki, open("evidence/robots_check_wikipedia.json", "w"), indent=2)

# --- 2.7 Komposisi data untuk analisis bias ---
komposisi = df["jenis_wisata"].str.split("/").str[0].value_counts()
print("Komposisi jenis utama:\n", komposisi.to_string())

{'robots_url': 'https://en.wikipedia.org/robots.txt', 'status': 200, 'boleh': True, 'crawl_delay': None, 'dicek': '2026-11-18T03:58:17'}
Komposisi jenis utama:
 jenis_wisata
Alam         68
Budaya       23
Sejarah       9
Buatan        3
Edukasi       2
Kerajinan     2
Seni          1


In [14]:
# --- 2.8 Teks tujuan, bias, dan retensi ---
TUJUAN = ("Menganalisis daya tarik wisata Bali per kabupaten/kota dengan profil wilayah dan kondisi cuaca 2026 "
          "untuk tugas UTS Data Acquisition & Management (INSTIKI); tujuan akademik non-komersial.")
CATATAN_BIAS = (
    f"(1) Daftar {len(df)} objek wisata adalah hasil kurasi dengan kuota seragam ({df['kabupaten_kota'].value_counts().iloc[0]} objek per kabupaten/kota), "
    f"bukan sensus seluruh daya tarik wisata Bali, sehingga tidak mewakili proporsi sebenarnya; kategori 'Alam' dominan ({int((df['jenis_wisata'].str.startswith('Alam')).sum())} objek). "
    "(2) Koordinat bersifat representatif kabupaten/kota, bukan per objek, jadi cuaca yang digabungkan adalah cuaca wilayah, bukan lokasi objek. "
    "(3) Cuaca Open-Meteo berasal dari reanalisis model berbasis grid, bukan stasiun pengamatan. "
    "(4) Status objek belum diverifikasi ke dinas/portal resmi.")
RETENSI = ("Data mentah, hasil olahan, metadata, dan bukti disimpan di repository tugas sampai penilaian semester selesai, "
           "lalu repository dijadikan privat/dihapus; tidak ada data pribadi yang disimpan. Data cuaca dan HTML Wikipedia tidak dipublikasikan ulang di luar tugas ini.")
print(CATATAN_BIAS)

(1) Daftar 108 objek wisata adalah hasil kurasi dengan kuota seragam (12 objek per kabupaten/kota), bukan sensus seluruh daya tarik wisata Bali, sehingga tidak mewakili proporsi sebenarnya; kategori 'Alam' dominan (68 objek). (2) Koordinat bersifat representatif kabupaten/kota, bukan per objek, jadi cuaca yang digabungkan adalah cuaca wilayah, bukan lokasi objek. (3) Cuaca Open-Meteo berasal dari reanalisis model berbasis grid, bukan stasiun pengamatan. (4) Status objek belum diverifikasi ke dinas/portal resmi.


In [15]:
# --- 2.9 Susun Data Ethics Checklist 9 butir ---
sudah_lisensi = all(s["lisensi"] for s in SUMBER)
bukti_ketentuan = (f"Wikipedia: robots.txt status {robots_wiki['status']}, boleh diakses={robots_wiki['boleh']}, "
                   f"crawl-delay={robots_wiki['crawl_delay']}, dicek {robots_wiki['dicek']}; "
                   "Open-Meteo: batas <10.000 panggilan/hari, non-komersial (halaman terms); "
                   "data.go.id: berkas dirujuk manual, tidak di-scrape, halaman dataset diperiksa manual.")
robots_ok = robots_wiki["boleh"] is True

checklist = pd.DataFrame([
    ("Sumber data dan URL dicantumkan", "Ya",
     "; ".join(f"{s['jenis']}: {s['url']}" for s in SUMBER) + " (rinci di metadata/provenance.csv)"),
    ("Lisensi data diperiksa dan dicatat", "Ya" if sudah_lisensi else "Tidak",
     "data.go.id: lisensi tidak dicantumkan pada halaman dataset -> dipakai konservatif (akademik, non-komersial, beratribusi); "
     "Open-Meteo: CC BY 4.0; Wikipedia: CC BY-SA 4.0. Bukti: " + "; ".join(bukti_lisensi)),
    ("robots.txt / Terms of Service diperiksa", "Ya" if robots_ok else "Tidak", bukti_ketentuan),
    ("Kolom PII diidentifikasi", "Ya",
     f"{len(klasifikasi)} kolom diperiksa (nama kolom + pola isi). PII langsung: {', '.join(pii_kolom) or 'tidak ada'}; "
     f"quasi-identifier: {', '.join(qi_kolom) or 'tidak ada'}. Alasan per kolom ada pada tabel 2.3."),
    ("PII dihapus, di-mask, atau di-pseudonimisasi",
     "Ya" if pii_kolom and sisa_pii.empty else ("NA" if not pii_kolom else "Tidak"),
     "; ".join(f"{k}: {a}" for k, a in TINDAKAN.items()) if pii_kolom
     else "Tidak ada kolom PII: data berupa daftar objek wisata (nama tempat) dan wilayah kabupaten/kota, bukan data individu."),
    ("Prinsip minimalisasi data diterapkan", "Ya",
     f"Dipakai {len(KOLOM_DIPAKAI)} kolom; dibuang: {', '.join(tidak_dipakai)} (konstan/keterangan, sudah dicatat di provenance)."),
    ("Potensi bias sampel dianalisis", "Ya" if CATATAN_BIAS.strip() else "Tidak", CATATAN_BIAS),
    ("Tujuan penggunaan data dinyatakan eksplisit", "Ya" if TUJUAN.strip() else "Tidak", TUJUAN),
    ("Rencana retensi dan penghapusan data ada", "Ya" if RETENSI.strip() else "Tidak", RETENSI),
], columns=["Aspek", "Status (Ya/Tidak/NA)", "Bukti / Catatan"])

def cek_kelengkapan(cl):
    """Pemeriksaan otomatis: 9 butir, status valid, bukti memadai, tidak ada butir 'Tidak'."""
    masalah = []
    if len(cl) != 9:
        masalah.append(f"Jumlah butir {len(cl)}, seharusnya 9")
    for _, r in cl.iterrows():
        status, bukti = r["Status (Ya/Tidak/NA)"], str(r["Bukti / Catatan"]).strip()
        if status not in ("Ya", "Tidak", "NA"):
            masalah.append(f"'{r['Aspek']}': status tidak valid ({status})")
        if status in ("Ya", "NA") and len(bukti) < 10:
            masalah.append(f"'{r['Aspek']}': status {status} tanpa bukti/alasan memadai")
        if status == "Tidak":
            masalah.append(f"'{r['Aspek']}': belum terpenuhi")
    return masalah

masalah = cek_kelengkapan(checklist)
print("Hasil pemeriksaan:", "LENGKAP" if not masalah else "BELUM LENGKAP")
for m in masalah:
    print(" -", m)
pd.set_option("display.max_colwidth", 200)
checklist

Hasil pemeriksaan: LENGKAP


,Aspek,Status (Ya/Tidak/NA),Bukti / Catatan
0,Sumber data dan URL dicantumkan,Ya,berkas: https://data.go.id/dataset/dataset/sebaran-daya-tarik-wisata-di-bali ; https://data.go.id/dataset/dataset/data-daya-tarik-wisata-alam-di-provinsi-bali; api: https://archive-api.open-meteo....
1,Lisensi data diperiksa dan dicatat,Ya,"data.go.id: lisensi tidak dicantumkan pada halaman dataset -> dipakai konservatif (akademik, non-komersial, beratribusi); Open-Meteo: CC BY 4.0; Wikipedia: CC BY-SA 4.0. Bukti: datagoid_sebaran_dt..."
2,robots.txt / Terms of Service diperiksa,Ya,"Wikipedia: robots.txt status 200, boleh diakses=True, crawl-delay=None, dicek 2026-11-18T03:58:17; Open-Meteo: batas <10.000 panggilan/hari, non-komersial (halaman terms); data.go.id: berkas diruj..."
3,Kolom PII diidentifikasi,Ya,9 kolom diperiksa (nama kolom + pola isi). PII langsung: tidak ada; quasi-identifier: tidak ada. Alasan per kolom ada pada tabel 2.3.
4,"PII dihapus, di-mask, atau di-pseudonimisasi",NA,"Tidak ada kolom PII: data berupa daftar objek wisata (nama tempat) dan wilayah kabupaten/kota, bukan data individu."
5,Prinsip minimalisasi data diterapkan,Ya,"Dipakai 6 kolom; dibuang: tahun_referensi_daftar, status_verifikasi, sumber_data (konstan/keterangan, sudah dicatat di provenance)."
6,Potensi bias sampel dianalisis,Ya,"(1) Daftar 108 objek wisata adalah hasil kurasi dengan kuota seragam (12 objek per kabupaten/kota), bukan sensus seluruh daya tarik wisata Bali, sehingga tidak mewakili proporsi sebenarnya; katego..."
7,Tujuan penggunaan data dinyatakan eksplisit,Ya,Menganalisis daya tarik wisata Bali per kabupaten/kota dengan profil wilayah dan kondisi cuaca 2026 untuk tugas UTS Data Acquisition & Management (INSTIKI); tujuan akademik non-komersial.
8,Rencana retensi dan penghapusan data ada,Ya,"Data mentah, hasil olahan, metadata, dan bukti disimpan di repository tugas sampai penilaian semester selesai, lalu repository dijadikan privat/dihapus; tidak ada data pribadi yang disimpan. Data ..."


In [16]:
# --- 2.10 Simpan checklist (CSV + Markdown) ---
checklist.to_csv("docs/data_ethics_checklist.csv", index=False)
with open("docs/data_ethics_checklist.md", "w", encoding="utf-8") as f:
    f.write(f"# Data Ethics Checklist\n\n{NAMA} ({NIM}) - UTS Data Acquisition & Management - {dt.date.today()}\n\n")
    f.write("| No | Aspek | Status | Bukti / Catatan |\n|---|---|---|---|\n")
    for i, r in checklist.iterrows():
        f.write(f"| {i+1} | {r['Aspek']} | {r['Status (Ya/Tidak/NA)']} | {str(r['Bukti / Catatan']).replace('|', '/')} |\n")
print("Tersimpan: docs/data_ethics_checklist.csv dan .md")

Tersimpan: docs/data_ethics_checklist.csv dan .md


In [17]:
# --- 3.1 Parameter: koordinat per kabupaten/kota diambil dari dataset, periode 2026 ---
koordinat = (df[["kabupaten_kota", "latitude_kabupaten", "longitude_kabupaten"]]
             .drop_duplicates().reset_index(drop=True))
START = "2026-01-01"
# Data arsip memiliki jeda pemrosesan beberapa hari -> akhiri 7 hari sebelum hari ini
END = min(dt.date.today() - dt.timedelta(days=7), dt.date(2026, 12, 31)).isoformat()
DAILY = ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
         "precipitation_sum", "wind_speed_10m_max"]          # variabel inti
DAILY_OPSIONAL = ["relative_humidity_2m_mean"]               # dicoba dulu; dibuang otomatis jika ditolak API (HTTP 400)
print(f"Periode: {START} s.d. {END}")
display(koordinat)

Periode: 2026-01-01 s.d. 2026-11-11


,kabupaten_kota,latitude_kabupaten,longitude_kabupaten
0,Badung,-8.65,115.18
1,Bangli,-8.45,115.35
2,Buleleng,-8.12,115.09
3,Denpasar,-8.67,115.22
4,Gianyar,-8.54,115.33
5,Jembrana,-8.35,114.65
6,Karangasem,-8.45,115.61
7,Klungkung,-8.55,115.43
8,Tabanan,-8.54,115.13


In [18]:
# --- 3.2 Ambil data per kabupaten/kota (jeda, timeout, retry ada di safe_get); simpan JSON mentah tanpa diubah ---
frames = []
for _, k in koordinat.iterrows():
    params = {"latitude": k["latitude_kabupaten"], "longitude": k["longitude_kabupaten"],
              "start_date": START, "end_date": END, "daily": ",".join(DAILY + DAILY_OPSIONAL), "timezone": "Asia/Makassar"}
    try:
        resp = safe_get(API_URL, params=params)
    except requests.HTTPError:                                  # variabel opsional ditolak -> ulangi dengan variabel inti saja
        print("    Variabel opsional ditolak API; mengulang dengan variabel inti.")
        DAILY_OPSIONAL = []
        params["daily"] = ",".join(DAILY)
        resp = safe_get(API_URL, params=params)
    path = f"data/raw/openmeteo_{k['kabupaten_kota'].lower()}_{START}_{END}.json"
    with open(path, "wb") as f:
        f.write(resp.content)                                   # respons mentah apa adanya
    log_provenance(f"Open-Meteo cuaca harian - {k['kabupaten_kota']}", "API", resp.url, path,
                   lisensi="CC BY 4.0 (atribusi: Weather data by Open-Meteo.com)",
                   catatan="Reanalisis berbasis grid; tanpa API key; zona waktu Asia/Makassar")
    # --- 3.3 Normalisasi JSON -> tabel ---
    js = resp.json()
    d = pd.DataFrame(js["daily"])
    d.insert(0, "kabupaten_kota", k["kabupaten_kota"])
    d["lat_grid"], d["lon_grid"] = js.get("latitude"), js.get("longitude")
    frames.append(d)
print("Selesai:", len(frames), "kabupaten/kota")

[1] HTTP 200 <- https://archive-api.open-meteo.com/v1/archive?latitude=-8.65&longitude=115.18&start_date=2026-01-01&end_date=2026-11-11&daily=temperature_2m_mean%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum%2Cwind_speed_10m_max%2Crelative_humidity_2m_mean&timezone=Asia%2FMakassar
[1] HTTP 200 <- https://archive-api.open-meteo.com/v1/archive?latitude=-8.45&longitude=115.35&start_date=2026-01-01&end_date=2026-11-11&daily=temperature_2m_mean%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum%2Cwind_speed_10m_max%2Crelative_humidity_2m_mean&timezone=Asia%2FMakassar
[1] HTTP 200 <- https://archive-api.open-meteo.com/v1/archive?latitude=-8.12&longitude=115.09&start_date=2026-01-01&end_date=2026-11-11&daily=temperature_2m_mean%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum%2Cwind_speed_10m_max%2Crelative_humidity_2m_mean&timezone=Asia%2FMakassar
[1] HTTP 200 <- https://archive-api.open-meteo.com/v1/archive?latitude=-8.67&longitude=115.22&start_date=2

In [19]:
# --- 3.4 Gabung & rapikan tabel cuaca harian ---
cuaca = pd.concat(frames, ignore_index=True).rename(columns={
    "time": "tanggal", "temperature_2m_mean": "suhu_rata2_c", "temperature_2m_max": "suhu_maks_c",
    "temperature_2m_min": "suhu_min_c", "relative_humidity_2m_mean": "kelembapan_persen",
    "precipitation_sum": "hujan_mm", "wind_speed_10m_max": "angin_maks_kmh"})
cuaca["tanggal"] = pd.to_datetime(cuaca["tanggal"])
if "kelembapan_persen" not in cuaca.columns:                    # API tidak menyediakan variabel ini
    cuaca["kelembapan_persen"] = np.nan

assert len(cuaca) >= 30, "Tabel API harus minimal 30 baris"
assert not cuaca.duplicated(["kabupaten_kota", "tanggal"]).any(), "Duplikat (kabupaten, tanggal)"
print("Baris:", len(cuaca), "| kabupaten:", cuaca['kabupaten_kota'].nunique(),
      "| rentang:", cuaca['tanggal'].min().date(), "s.d.", cuaca['tanggal'].max().date())
print("Nilai kosong per kolom:\n", cuaca.isna().sum().to_string())
cuaca.to_csv("data/processed/cuaca_harian_bali_2026.csv", index=False)
cuaca.head()

Baris: 2835 | kabupaten: 9 | rentang: 2026-01-01 s.d. 2026-11-11
Nilai kosong per kolom:
 kabupaten_kota       0
tanggal              0
suhu_rata2_c         0
suhu_maks_c          0
suhu_min_c           0
hujan_mm             0
angin_maks_kmh       0
kelembapan_persen    0
lat_grid             0
lon_grid             0


,kabupaten_kota,tanggal,suhu_rata2_c,suhu_maks_c,suhu_min_c,hujan_mm,angin_maks_kmh,kelembapan_persen,lat_grid,lon_grid
0,Badung,2026-01-01,28.4,32.7,24.8,0.6,9.9,72,-8.681898,115.193794
1,Badung,2026-01-02,27.5,31.3,25.3,2.8,12.9,80,-8.681898,115.193794
2,Badung,2026-01-03,27.7,31.5,24.6,0.6,10.4,74,-8.681898,115.193794
3,Badung,2026-01-04,27.2,32.4,23.1,3.4,16.3,76,-8.681898,115.193794
4,Badung,2026-01-05,26.1,28.7,24.1,9.9,19.8,85,-8.681898,115.193794


In [20]:
# --- 4.1 Bukti kepatuhan robots.txt (pemeriksaan segar; salinan robots.txt tersimpan di evidence/) ---
robots = cek_robots(PAGE_URL)
bukti_robots = {**robots, "page_url": PAGE_URL, "user_agent": HEADERS["User-Agent"],
                "kesimpulan": "BOLEH di-scrape" if robots["boleh"] else "TIDAK boleh / tidak pasti"}
json.dump(bukti_robots, open("evidence/robots_check.json", "w"), indent=2)
print(json.dumps(bukti_robots, indent=2))
assert robots["boleh"] is True, "robots.txt tidak mengizinkan / tidak dapat diperiksa -> berhenti"

# Simpan juga robots.txt sebagai berkas mentah untuk provenance
robots_path = "evidence/robots_" + urlparse(PAGE_URL).netloc.replace(".", "_") + ".txt"
log_provenance("robots.txt Wikipedia (bukti kepatuhan)", "Scraping (bukti)", robots["robots_url"], robots_path,
               lisensi="-", catatan="Dasar keputusan scraping; lihat evidence/robots_check.json")

{
  "robots_url": "https://en.wikipedia.org/robots.txt",
  "status": 200,
  "boleh": true,
  "crawl_delay": null,
  "dicek": "2026-11-18T03:58:52",
  "page_url": "https://en.wikipedia.org/wiki/List_of_districts_in_Bali",
  "user_agent": "INSTIKI-UTS-DAM/1.0 (mahasiswa; igustingurahdwiandikaputra@instiki.ac.id)",
  "kesimpulan": "BOLEH di-scrape"
}


In [21]:
# --- 4.2 Ambil halaman statis & simpan HTML mentah ---
r_page = safe_get(PAGE_URL)
PAGE_RAW = "data/raw/wikipedia_list_of_districts_in_bali.html"
with open(PAGE_RAW, "wb") as f:
    f.write(r_page.content)
log_provenance("Wikipedia - List of districts in Bali", "Scraping", PAGE_URL, PAGE_RAW,
               lisensi="CC BY-SA 4.0 (wajib atribusi)", catatan="HTML mentah; robots.txt diperiksa dan diizinkan")

[1] HTTP 200 <- https://en.wikipedia.org/wiki/List_of_districts_in_Bali


In [22]:
# --- 4.3 Ekstrak tabel ringkasan kabupaten/kota dengan pd.read_html ---
# Baca SEMUA tabel di halaman, lalu pilih tabel yang benar dari ISI-nya (bukan dari teks header yang bisa berubah-ubah)
semua = pd.read_html(io.StringIO(r_page.text), thousands=",")
print("Jumlah tabel di halaman:", len(semua))

def bersihkan(seri):
    """Ubah ke teks bersih: buang spasi khusus (nbsp) dan catatan kaki seperti [a]."""
    return (seri.astype(str).str.replace("\xa0", " ", regex=False)
            .str.replace(r"\[.*?\]", "", regex=True).str.strip())

def kolom_nama_wilayah(t):
    """Indeks kolom yang berisi >= 9 nilai berakhiran 'Regency' atau 'City'; None jika bukan tabel target."""
    for i in range(t.shape[1]):
        if bersihkan(t.iloc[:, i]).str.contains(r"(?:Regency|City)$", regex=True).sum() >= 9:
            return i
    return None

tabel, idx = None, None
for t in semua:
    i = kolom_nama_wilayah(t)
    if i is not None:
        tabel, idx = t, i
        break
assert tabel is not None, "Tabel kabupaten/kota tidak ditemukan; periksa struktur halaman"
print("Tabel target ditemukan; kolom nama berada di indeks", idx)
display(tabel.head(3))

# Kolom di sebelah kanan kolom nama: luas, penduduk, kepadatan, jumlah kecamatan (urutan sesuai halaman)
nama = bersihkan(tabel.iloc[:, idx])
baris = nama.str.contains(r"(?:Regency|City)$", regex=True)          # buang baris total
def angka(offset):
    return pd.to_numeric(bersihkan(tabel.loc[baris].iloc[:, idx + offset]).str.replace(",", "", regex=False), errors="coerce")

profil = pd.DataFrame({
    "kabupaten_kota": nama[baris].str.replace(r"\s+(Regency|City)$", "", regex=True).str.strip(),
    "luas_km2": angka(1),
    "penduduk_2020": angka(2),
    "kepadatan_per_km2": angka(3),
    "jumlah_kecamatan": angka(4),
}).reset_index(drop=True)

# Validasi: 9 baris, nama unik, nilai masuk akal (mendeteksi kolom yang salah posisi)
assert len(profil) == 9, f"Seharusnya 9 kabupaten/kota, ditemukan {len(profil)}"
assert profil["kabupaten_kota"].is_unique
assert profil["luas_km2"].between(100, 1500).all(), "Kolom luas tidak sesuai"
assert (profil["penduduk_2020"] > 100_000).all(), "Kolom penduduk tidak sesuai"
profil.to_csv("data/processed/profil_kabupaten_bali_wikipedia.csv", index=False)
profil

Jumlah tabel di halaman: 11
Tabel target ditemukan; kolom nama berada di indeks 2


No. Code Kemendagri  Regency  or City Area (km²) Population (2020)  \
   No. Code Kemendagri  Regency  or City Area (km²) Population (2020)   
0  1.0           51.03    Badung Regency     418.62            548191   
1  2.0           51.06    Bangli Regency     490.71            258191   
2  3.0           51.08  Buleleng Regency    1364.73            791813   

  Density (people/km²) Comprises                              
  Density (people/km²)  District Urban village Rural village  
0              1309.52         6            16            46  
1               526.16         4             4            68  
2               580.20         9            19           129

,kabupaten_kota,luas_km2,penduduk_2020,kepadatan_per_km2,jumlah_kecamatan
0,Badung,418.62,548191,1309.52,6
1,Bangli,490.71,258191,526.16,4
2,Buleleng,1364.73,791813,580.20,9
3,Gianyar,368.00,515344,1403.11,7
4,Jembrana,841.80,317064,376.65,5
5,Karangasem,839.54,492402,586.51,8
6,Klungkung,315.00,206925,656.90,4
7,Tabanan,1013.88,461630,455.32,10
8,Denpasar,127.78,725314,5676.27,4


In [23]:
# --- 5.1 Ringkas cuaca harian -> satu baris per kabupaten/kota ---
cuaca_kab = (cuaca.groupby("kabupaten_kota")
             .agg(suhu_rata2_c=("suhu_rata2_c", "mean"),
                  suhu_maks_rata2_c=("suhu_maks_c", "mean"),
                  suhu_min_rata2_c=("suhu_min_c", "mean"),
                  kelembapan_rata2_persen=("kelembapan_persen", "mean"),
                  hujan_total_mm=("hujan_mm", "sum"),
                  hari_hujan=("hujan_mm", lambda s: int((s >= 1).sum())),
                  angin_maks_kmh=("angin_maks_kmh", "max"),
                  jumlah_hari_cuaca=("tanggal", "nunique"))
             .round(2).reset_index())
assert cuaca_kab["kabupaten_kota"].is_unique

# --- 5.2 Pastikan penulisan nama kabupaten/kota konsisten di ketiga sumber ---
nama_wisata, nama_cuaca, nama_profil = (set(df["kabupaten_kota"]), set(cuaca_kab["kabupaten_kota"]), set(profil["kabupaten_kota"]))
print("Selisih wisata-cuaca :", nama_wisata ^ nama_cuaca or "tidak ada")
print("Selisih wisata-profil:", nama_wisata ^ nama_profil or "tidak ada")

Selisih wisata-cuaca : tidak ada
Selisih wisata-profil: tidak ada


In [24]:
# --- 5.3 Siapkan tabel wisata (kolom minimal) + rapikan kategori jenis wisata ---
wisata = aman[KOLOM_DIPAKAI].copy()
# 'Alam/Budaya' dan 'Budaya/Alam' dianggap kombinasi yang sama -> urutkan token
wisata["jenis_normal"] = wisata["jenis_wisata"].map(lambda s: "/".join(sorted(x.strip() for x in s.split("/"))))
n_wisata = len(wisata)

# --- 5.4 Merge dengan validate: wisata -> cuaca (m:1) -> profil Wikipedia (m:1) ---
gabungan = (wisata
            .merge(cuaca_kab, on="kabupaten_kota", how="left", validate="m:1")
            .merge(profil, on="kabupaten_kota", how="left", validate="m:1"))

# --- 5.5 Bukti tidak ada duplikasi / baris hilang ---
bukti_merge = {
    "baris_wisata_sebelum": n_wisata,
    "baris_gabungan_sesudah": len(gabungan),
    "id_wisata_unik": bool(gabungan["id_wisata"].is_unique),
    "baris_duplikat_penuh": int(gabungan.duplicated().sum()),
    "baris_tanpa_pasangan_cuaca": int(gabungan["suhu_rata2_c"].isna().sum()),
    "baris_tanpa_pasangan_profil": int(gabungan["luas_km2"].isna().sum()),
}
print(json.dumps(bukti_merge, indent=2))
assert len(gabungan) == n_wisata, "Jumlah baris berubah -> ada duplikasi!"
assert gabungan["id_wisata"].is_unique and gabungan.duplicated().sum() == 0
assert bukti_merge["baris_tanpa_pasangan_cuaca"] == 0 and bukti_merge["baris_tanpa_pasangan_profil"] == 0
gabungan.head()

{
  "baris_wisata_sebelum": 108,
  "baris_gabungan_sesudah": 108,
  "id_wisata_unik": true,
  "baris_duplikat_penuh": 0,
  "baris_tanpa_pasangan_cuaca": 0,
  "baris_tanpa_pasangan_profil": 0
}


,id_wisata,nama_objek_wisata,kabupaten_kota,jenis_wisata,latitude_kabupaten,longitude_kabupaten,jenis_normal,suhu_rata2_c,suhu_maks_rata2_c,suhu_min_rata2_c,kelembapan_rata2_persen,hujan_total_mm,hari_hujan,angin_maks_kmh,jumlah_hari_cuaca,luas_km2,penduduk_2020,kepadatan_per_km2,jumlah_kecamatan
0,BALI001,Pantai Kuta,Badung,Alam,-8.65,115.18,Alam,26.13,29.34,23.2,80.77,1517.1,189,32.8,315,418.62,548191,1309.52,6
1,BALI002,Pantai Legian,Badung,Alam,-8.65,115.18,Alam,26.13,29.34,23.2,80.77,1517.1,189,32.8,315,418.62,548191,1309.52,6
2,BALI003,Pantai Seminyak,Badung,Alam,-8.65,115.18,Alam,26.13,29.34,23.2,80.77,1517.1,189,32.8,315,418.62,548191,1309.52,6
3,BALI004,Pantai Petitenget,Badung,Alam,-8.65,115.18,Alam,26.13,29.34,23.2,80.77,1517.1,189,32.8,315,418.62,548191,1309.52,6
4,BALI005,Pantai Berawa,Badung,Alam,-8.65,115.18,Alam,26.13,29.34,23.2,80.77,1517.1,189,32.8,315,418.62,548191,1309.52,6


In [25]:
# --- 5.6 Simpan hasil + metadata ---
OUT = "data/processed/gabungan_wisata_cuaca_profil_bali.csv"
gabungan.to_csv(OUT, index=False, encoding="utf-8")

meta = {
    "judul": "Gabungan daya tarik wisata Bali + cuaca harian 2026 (ringkasan) + profil kabupaten/kota",
    "pembuat": f"{NAMA} ({NIM})", "dibuat": dt.datetime.now().isoformat(timespec="seconds"),
    "berkas_hasil": OUT, "md5": md5_file(OUT),
    "jumlah_baris": len(gabungan), "jumlah_kolom": gabungan.shape[1],
    "kolom": {c: str(t) for c, t in gabungan.dtypes.items()},
    "metode_gabung": {"kunci": "kabupaten_kota", "urutan": ["wisata -> cuaca (left, m:1)", "-> profil Wikipedia (left, m:1)"],
                      "catatan": "Cuaca harian diringkas per kabupaten/kota karena dataset wisata tidak memiliki tanggal kunjungan."},
    "bukti_tanpa_duplikasi": bukti_merge,
    "periode_cuaca": {"mulai": START, "selesai": END},
    "sumber": list(provenance.values()),
    "atribusi": ["Weather data by Open-Meteo.com (CC BY 4.0)", "Wikipedia 'List of districts in Bali' (CC BY-SA 4.0)",
                 "Daftar objek wisata merujuk data.go.id (Pemprov Bali); lisensi tidak dicantumkan"],
    "keterbatasan": CATATAN_BIAS,
}
json.dump(meta, open("metadata/gabungan_metadata.json", "w"), indent=2, ensure_ascii=False, default=str)
print("Tersimpan:", OUT, "dan metadata/gabungan_metadata.json")

Tersimpan: data/processed/gabungan_wisata_cuaca_profil_bali.csv dan metadata/gabungan_metadata.json


In [26]:
# --- 6.1 Daftarkan berkas lain di data/raw yang belum tercatat (mis. berkas asli portal yang diunduh manual) ---
for p in sorted(Path("data/raw").glob("*")):
    if str(p).replace("\\", "/") not in provenance and p.is_file():
        log_provenance(f"Berkas tambahan {p.name}", "Berkas (diunduh manual)", f"{URL_DTW_1} ; {URL_DTW_2}", str(p),
                       lisensi="Tidak dicantumkan pada halaman dataset", catatan="Ditambahkan otomatis; isi manual bila perlu")

# --- 6.2 Verifikasi: semua berkas raw utuh (MD5 sama dengan saat dicatat) ---
prov = pd.read_csv(PROV_PATH)
for _, row in prov.iterrows():
    assert md5_file(row["berkas_raw"]) == row["md5"], f"Berkas berubah: {row['berkas_raw']}"
print("Semua berkas raw utuh (MD5 cocok).")
prov[["sumber", "cara_akuisisi", "tanggal_akses", "lisensi", "berkas_raw", "md5"]]

Semua berkas raw utuh (MD5 cocok).


,sumber,cara_akuisisi,tanggal_akses,lisensi,berkas_raw,md5
0,"Daftar objek wisata Bali (berkas XLSX, sheet Wisata_108)",Berkas (diunduh/disusun manual),2026-11-18,Tidak dicantumkan pada halaman dataset data.go.id (diperiksa pada tanggal akses),data/raw/dataset_wisata_cuaca_bali_2026.xlsx,47152b5e47620f3a4628bdacdaf2def4
1,Open-Meteo cuaca harian - Badung,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_badung_2026-01-01_2026-11-11.json,ab6b9c1e01e25429514fbc268f1c946c
2,Open-Meteo cuaca harian - Bangli,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_bangli_2026-01-01_2026-11-11.json,835c94930db349fb14a859adce749cf2
3,Open-Meteo cuaca harian - Buleleng,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_buleleng_2026-01-01_2026-11-11.json,e3a44668855e7764c9a4d62f88d0848f
4,Open-Meteo cuaca harian - Denpasar,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_denpasar_2026-01-01_2026-11-11.json,89427ddb65f552b565c5deedcd2bf510
5,Open-Meteo cuaca harian - Gianyar,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_gianyar_2026-01-01_2026-11-11.json,5f3ce2a8957e22a49cce9ea536d78f3b
6,Open-Meteo cuaca harian - Jembrana,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_jembrana_2026-01-01_2026-11-11.json,332cecda17c76a5ce5e201a34d94ea1e
7,Open-Meteo cuaca harian - Karangasem,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_karangasem_2026-01-01_2026-11-11.json,8913d306d28e1b46cf007633ecd576d4
8,Open-Meteo cuaca harian - Klungkung,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_klungkung_2026-01-01_2026-11-11.json,cff9abdb4e880f8f669f487f767f1aa4
9,Open-Meteo cuaca harian - Tabanan,API,2026-11-18,CC BY 4.0 (atribusi: Weather data by Open-Meteo.com),data/raw/openmeteo_tabanan_2026-01-01_2026-11-11.json,7833f2e0566a50098d620fa9d2dbc8ab


In [27]:
# --- 6.3 Paket pengumpulan: zip semua folder, lalu (Colab) unduh ---
ARSIP = "UTS_DAM_2501010016"
shutil.make_archive(ARSIP, "zip", root_dir=".", base_dir="data")        # data/raw + data/processed
import zipfile
with zipfile.ZipFile(f"{ARSIP}.zip", "a") as z:
    for folder in ["metadata", "docs", "evidence"]:
        for p in Path(folder).rglob("*"):
            if p.is_file():
                z.write(p)
print("Arsip:", f"{ARSIP}.zip", f"({os.path.getsize(ARSIP + '.zip')/1024:.1f} KB)")
try:
    from google.colab import files
    files.download(f"{ARSIP}.zip")
except ImportError:
    pass

Arsip: UTS_DAM_2501010016.zip (626.8 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>